In [1]:
!pip install pandas scikit-learn

In [2]:
import pandas as pd
import torch
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import os

# --- 1. CONFIGURATION ---
NODES_CSV_PATH = "merged_nodes.csv"
MODEL_NAME = "FremyCompany/BioLORD-2023"
THRESHOLD = 0.75 # Higher threshold for better precision

class SmartMedicalMatcher:
    def __init__(self, csv_path):
        print(f"--- Initializing Smart Matcher with BioLORD ---")
        
        # 1. Load BioLORD (Should be fast if already cached)
        self.embeddings_model = HuggingFaceEmbeddings(
            model_name=MODEL_NAME,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )

        # 2. Load KG Nodes
        df = pd.read_csv(csv_path, encoding='latin-1')
        possible_cols = ['source_name']
        target_col = next((c for c in possible_cols if c in df.columns), df.columns[0])
        
        self.kg_names = df[target_col].astype(str).tolist()
        
        # 3. Pre-calculate KG Embeddings (This makes matching fast)
        # We process in batches to avoid memory issues
        print(f"🧠 Encoding {len(self.kg_names)} KG nodes into semantic space...")
        self.kg_vectors = self.embeddings_model.embed_documents(self.kg_names)
        self.kg_vectors = np.array(self.kg_vectors)
        print("✅ Semantic Space Ready.")

    def match(self, emr_term: str):
        # 1. Expand common abbreviations before matching
        abbrev_map = {
            "hb": "hemoglobin",
            "uti": "urinary tract infection",
            "bp": "blood pressure",
            "wbc": "white blood cell count",
            "mi": "myocardial infarction",
            "ckd": "chronic kidney disease"
        }
        
        term = emr_term.lower().strip()
        for abbrev, full in abbrev_map.items():
            if abbrev == term or abbrev in term.split():
                term = term.replace(abbrev, full)

        # 2. Get Embedding for the term
        term_vector = np.array(self.embeddings_model.embed_query(term)).reshape(1, -1)

        # 3. Calculate Cosine Similarity
        similarities = cosine_similarity(term_vector, self.kg_vectors)[0]
        best_idx = np.argmax(similarities)
        score = similarities[best_idx]

        if score >= THRESHOLD:
            return {
                "term": emr_term,
                "match": self.kg_names[best_idx],
                "conf": round(float(score), 3)
            }
        else:
            return {"term": emr_term, "match": "UNRESOLVED", "conf": round(float(score), 3)}

# --- EXECUTION ---
if __name__ == "__main__":
    matcher = SmartMedicalMatcher(NODES_CSV_PATH)

    test_cases = ["heart attack", "hb low", "kidney fail", "glucose high", "uti info"]

    print("\n" + "="*50)
    print(f"{'EMR TERM':<20} | {'BIO-LORD MATCH':<25} | {'CONF'}")
    print("-" * 55)
    for t in test_cases:
        res = matcher.match(t)
        print(f"{res['term']:<20} | {res['match']:<25} | {res['conf']}")
    print("="*50)

--- Initializing Smart Matcher with BioLORD ---


OSError: FremyCompany/BioLORD-2023 does not appear to have a file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt or flax_model.msgpack.

In [3]:
import os
import pickle
import glob
import numpy as np
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings

# --- CONFIGURATION ---
NODES_CSV_PATH = "merged_nodes.csv"
VECTORS_SAVE_PATH = "kg_vectors.npy"
NAMES_SAVE_PATH = "kg_names.pkl"

# 1. FIX: Point to your local D: drive path instead of the internet
HF_HUB_PATH = r"D:\huggingface\hub\models--FremyCompany--BioLORD-2023\snapshots"
snapshot_folders = glob.glob(os.path.join(HF_HUB_PATH, "*"))

if not snapshot_folders:
    raise FileNotFoundError(f"❌ Could not find BioLORD files in {HF_HUB_PATH}. Check your D: drive path.")

# This automatically picks the long ID folder (e.g. 167aab52...)
LOCAL_MODEL_PATH = snapshot_folders[0]

def save_kg_embeddings():
    print("--- Starting Embedding Save Process ---")
    print(f"📍 Using Local Model: {LOCAL_MODEL_PATH}")
    
    # 1. Load the CSV
    try:
        # Using 'utf-8' but falling back to 'latin-1' if needed
        df = pd.read_csv(NODES_CSV_PATH, encoding='utf-8')
    except:
        df = pd.read_csv(NODES_CSV_PATH, encoding='latin-1')
        
    # Find the name column
    target_col = next((c for c in ['node_name', 'name', 'LABEL'] if c in df.columns), df.columns[0])
    kg_names = df[target_col].astype(str).tolist()
    print(f"📄 Found {len(kg_names)} nodes in CSV.")

    # 2. Initialize BioLORD from LOCAL PATH
    print(f"🧠 Initializing BioLORD (Local Offline Mode)...")
    embeddings_model = HuggingFaceEmbeddings(
        model_name=LOCAL_MODEL_PATH,
        model_kwargs={
            'device': 'cpu',
            'local_files_only': True # Prevents it from trying to check the internet
        },
        encode_kwargs={'normalize_embeddings': True}
    )

    # 3. Generate Embeddings
    print("⏳ Encoding nodes into vectors... (This may take a few minutes)")
    # We use embed_documents to process the list
    kg_vectors = np.array(embeddings_model.embed_documents(kg_names))

    # 4. SAVE TO DISK
    print("💾 Saving files to disk...")
    np.save(VECTORS_SAVE_PATH, kg_vectors)
    with open(NAMES_SAVE_PATH, 'wb') as f:
        pickle.dump(kg_names, f)

    print(f"✅ SUCCESS!")
    print(f"Saved: {VECTORS_SAVE_PATH} ({os.path.getsize(VECTORS_SAVE_PATH) // 1024**2} MB)")
    print(f"Saved: {NAMES_SAVE_PATH}")

def load_kg_embeddings():
    print("\n🚀 Testing Load Speed...")
    import time
    start = time.time()
    
    vectors = np.load(VECTORS_SAVE_PATH)
    with open(NAMES_SAVE_PATH, 'rb') as f:
        names = pickle.load(f)
        
    end = time.time()
    print(f"✅ Loaded {len(names)} vectors in {round(end - start, 4)} seconds!")
    return vectors, names

# --- EXECUTION ---
if __name__ == "__main__":
    if not os.path.exists(VECTORS_SAVE_PATH):
        save_kg_embeddings()
    else:
        print("💡 Embedding files already exist. Loading them now...")
    
    load_kg_embeddings()

--- Starting Embedding Save Process ---
📍 Using Local Model: D:\huggingface\hub\models--FremyCompany--BioLORD-2023\snapshots\167aab527b238a50ca65224e6319215d2ff4fc9f
📄 Found 8729 nodes in CSV.
🧠 Initializing BioLORD (Local Offline Mode)...
⏳ Encoding nodes into vectors... (This may take a few minutes)
💾 Saving files to disk...
✅ SUCCESS!
Saved: kg_vectors.npy (51 MB)
Saved: kg_names.pkl

🚀 Testing Load Speed...
✅ Loaded 8729 vectors in 0.0473 seconds!
